In [4]:
import pandas as pd

In [5]:
df_movies = pd.read_csv("../data/raw/movies.csv")
df_ratings = pd.read_csv("../data/raw/ratings.csv")

In [6]:
df_movies.info()
print(40 * "#")
df_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB
########################################
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     105339 non-null  int64  
 1   movieId    105339 non-null  int64  
 2   rating     105339 non-null  float64
 3   timestamp  105339 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.2 MB


In [7]:
df_ratings["timestamp"] = pd.to_datetime(df_ratings["timestamp"], unit="s")

In [8]:
# Extract hour
hours = df_ratings["timestamp"].dt.hour

# Define bins and labels
bins = [0, 6, 18, 24]  # Madrugada: 0-5, Dia: 6-17, Noite: 18-23
labels = ["Madrugada", "Dia", "Noite"]

# Create categorical column
df_ratings["periodo"] = pd.cut(hours, bins=bins, labels=labels, right=False)

In [10]:
one_hot = df_movies["genres"].str.get_dummies(sep="|")
df_movies_encoded = pd.concat([df_movies[["movieId", "title"]], one_hot], axis=1)

In [11]:
df_movies_encoded.head()

,movieId,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
df_ratings = df_ratings.drop(columns="timestamp")

In [13]:
df_ratings.head()

,userId,movieId,rating,periodo
0,1,16,4.0,Madrugada
1,1,24,1.5,Madrugada
2,1,32,4.0,Madrugada
3,1,47,4.0,Madrugada
4,1,50,4.0,Madrugada


In [14]:
periodo_order = {"Madrugada": 0, "Dia": 1, "Noite": 2}
df_ratings["period"] = df_ratings["periodo"].map(periodo_order)

In [15]:
df_ratings

,userId,movieId,rating,periodo,period
0,1,16,4.0,Madrugada,0
1,1,24,1.5,Madrugada,0
2,1,32,4.0,Madrugada,0
3,1,47,4.0,Madrugada,0
4,1,50,4.0,Madrugada,0
...,...,...,...,...,...
105334,668,142488,4.0,Madrugada,0
105335,668,142507,3.5,Madrugada,0
105336,668,143385,4.0,Dia,1
105337,668,144976,2.5,Noite,2


In [16]:
df_merged = pd.merge(df_ratings, df_movies_encoded, on="movieId", how="inner")

In [17]:
df_merged = df_merged.drop(columns=["periodo", "(no genres listed)"])

In [18]:
df_merged

,userId,movieId,rating,period,title,Action,Adventure,Animation,Children,Comedy,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,16,4.0,0,Casino (1995),0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,24,1.5,0,Powder (1995),0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,1,32,4.0,0,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0,0,0,0,0,...,0,0,0,0,1,0,1,1,0,0
3,1,47,4.0,0,Seven (a.k.a. Se7en) (1995),0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,1,50,4.0,0,"Usual Suspects, The (1995)",0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105334,668,142488,4.0,0,Spotlight (2015),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
105335,668,142507,3.5,0,Pawn Sacrifice (2015),0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
105336,668,143385,4.0,1,Bridge of Spies (2015),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
105337,668,144976,2.5,2,Bone Tomahawk (2015),0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1


In [87]:
df_merged.to_csv("../data/processed/movies_ratings.csv")